# 03 · Special methods

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 1 | about 45 minutes live | [02 · Inheritance](02_inheritance.ipynb) | [04 · Encapsulation and abstraction](04_encapsulation_and_abstraction.ipynb) |

Why does `len(df)` work on a DataFrame? Why can you write `tn, fp, fn, tp = confusion_matrix(y, p).ravel()`, or `pipeline[0]` on a scikit-learn `Pipeline`? None of this is magic. Python translates the syntax into calls of methods with double underscores, such as `__len__` or `__getitem__`. When your own classes define these methods, they plug into the same syntax.

The theory parts use small, self-contained examples: a confusion matrix, the list of features a model uses, and a cost metric. In the hands-on parts you apply them to the churn model: cross-validation results, monthly batches and a business cost for the retention campaign.

> **Learning goals.** By the end you can:
>
> - explain what a special ("dunder") method is and which syntax triggers which method
> - give a class a useful `__repr__`, and make it work with `len()`, `[]`, `for`, `==` and `+`
> - write a callable object with `__call__` and use it wherever a function is expected
> - judge when overloading an operator helps readers and when it hides meaning

**Before you start:** notebooks 01 (classes) and 02 (inheritance).

**In this notebook**

- Theory 1 · What special methods are; `__repr__` and `__str__`
- Theory 2 · Containers: `__len__`, `__getitem__`, `__contains__`, `__iter__`
- Theory 3 · Equality, arithmetic, and when not to overload
- Hands-on A · Cross-validation results for the churn model
- Theory 4 · Callable objects and context managers
- Hands-on B · A business cost for the retention campaign

---

## Setup

Data and a simple churn model for the hands-on parts. The model is deliberately simple (a logistic regression on five columns): the point of this notebook is the classes around it, not the model. The theory parts do not need any of this.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("../data/raw/telco_churn.csv")

raw = pd.read_csv(DATA_PATH)
# TotalCharges is text and blank for customers who joined this month (tenure 0): they owe 0 so far.
# str.strip() removes spaces, to_numeric turns text into numbers (blanks become NaN), fillna(0.0) replaces NaN
total_charges = pd.to_numeric(raw["TotalCharges"].str.strip(), errors="coerce").fillna(0.0)

X = pd.DataFrame(
    {
        "tenure": raw["tenure"],
        "MonthlyCharges": raw["MonthlyCharges"],
        "TotalCharges": total_charges,
        "SeniorCitizen": raw["SeniorCitizen"],
        # (column == value) is True/False per row; astype(int) turns it into 1/0
        "month_to_month": (raw["Contract"] == "Month-to-month").astype(int),
    }
).set_index(raw["customerID"])  # rows are labelled by customer ID instead of 0, 1, 2, ...
y = pd.Series((raw["Churn"] == "Yes").astype(int).values, index=X.index, name="churn")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
# predict_proba gives one column per class; [:, 1] keeps the second column: the probability of churn
churn_proba = model.predict_proba(X_test)[:, 1]
y_pred = (churn_proba >= 0.5).astype(int)  # 1 where the probability is at least 0.5
print(f"{len(X_test)} test customers, {y_test.sum()} of them churned")

---

## Theory 1 · What special methods are; `__repr__` and `__str__`

A special method is a method whose name starts and ends with two underscores ("dunder" = double underscore). You almost never call them yourself. Python calls them when you use a piece of syntax on your object:

| You write | Python calls |
|---|---|
| `obj` in the REPL or last line of a cell, `repr(obj)` | `obj.__repr__()` |
| `print(obj)`, `str(obj)`, f-string `f"{obj}"` | `obj.__str__()` (falls back to `__repr__`) |
| `len(obj)` | `obj.__len__()` |
| `obj[key]` | `obj.__getitem__(key)` |
| `key in obj` | `obj.__contains__(key)` (falls back to iterating) |
| `for x in obj`, `a, b = obj`, `list(obj)` | `obj.__iter__()` |
| `a == b` | `a.__eq__(b)` |
| `a + b` | `a.__add__(b)`, and if that gives up, `b.__radd__(a)` |
| `obj(...)` | `obj.__call__(...)` |
| `with obj as x:` | `obj.__enter__()` and later `obj.__exit__(...)` |
| `if obj:` | `obj.__bool__()` (falls back to `__len__`) |

You already met one: `__init__` runs when you write `ConfusionMatrix(...)`. The full list is called the Python **data model**. pandas, NumPy and scikit-learn are built on it, which is why their objects feel like built-in types.

Here is the `ConfusionMatrix` from notebook 01 in compact form. The constructor takes the four counts in the order scikit-learn uses (`tn, fp, fn, tp`).

In [ ]:
class ConfusionMatrix:
    """Counts of a binary classifier's hits and misses (1 = positive class)."""

    def __init__(self, tn: int, fp: int, fn: int, tp: int):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0


cm = ConfusionMatrix(tn=900, fp=135, fn=170, tp=204)
cm

That last line shows something like `<__main__.ConfusionMatrix at 0x7f...>`. Correct, but useless. Two special methods fix it:

- `__repr__` is for developers: it should say unambiguously what the object is. Convention: look like the code that would create it.
- `__str__` is for readers of reports: whatever is most readable. If you define only `__repr__`, `print()` uses it too.

In [ ]:
class ConfusionMatrix:
    """Counts of a binary classifier's hits and misses (1 = positive class)."""

    def __init__(self, tn: int, fp: int, fn: int, tp: int):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0

    def __repr__(self) -> str:
        return f"ConfusionMatrix(tn={self.tn}, fp={self.fp}, fn={self.fn}, tp={self.tp})"

    def __str__(self) -> str:
        return (
            f"              pred=0  pred=1\n"
            f"actual=0  {self.tn:>8} {self.fp:>7}\n"
            f"actual=1  {self.fn:>8} {self.tp:>7}"
        )


cm = ConfusionMatrix(tn=900, fp=135, fn=170, tp=204)
print(cm)  # __str__
print()
print(repr(cm))  # __repr__
cm  # the notebook displays the last expression with __repr__

A good `__repr__` pays off every time you debug: in error messages, in the variable explorer, in a list of objects (`[cm, cm]` uses `__repr__` for each element).

> **Check your understanding**
>
> ```python
> results = [ConfusionMatrix(1, 2, 3, 4)]
> print(results)
> ```
>
> Does this print the table from `__str__` or the one-liner from `__repr__`?
>
> <details><summary><b>Show answer</b></summary>
>
> The one-liner: `[ConfusionMatrix(tn=1, fp=2, fn=3, tp=4)]`. Containers always show their elements with `repr()`, even when you `print` the container. That is one more reason to always write a `__repr__` first.
> </details>

In [ ]:
# Predict the output first, then run this cell
results = [ConfusionMatrix(1, 2, 3, 4)]
print(results)

---

## Theory 2 · Containers: `__len__`, `__getitem__`, `__contains__`, `__iter__`

Objects that hold several items (a list of fold scores, the customers in a file, the features of a model) become much easier to use when they answer the usual container questions: *how many?*, *give me item i*, *is this one in there?*, *let me loop over them*.

Example: the input columns a model was trained on, in order. Each special method simply **delegates** to the list stored inside.

In [ ]:
class FeatureSet:
    """The input columns a model was trained on, in order."""

    def __init__(self, names: list[str]):
        self.names = list(names)

    def __len__(self) -> int:
        return len(self.names)

    def __getitem__(self, position: int) -> str:
        return self.names[position]

    def __contains__(self, name: str) -> bool:
        return name in self.names

    def __iter__(self):
        return iter(self.names)  # iter(...) hands out the items one by one, here: the list's items

    def __repr__(self) -> str:
        return f"FeatureSet({self.names})"


features = FeatureSet(["tenure", "MonthlyCharges", "Contract"])
print(features)
print(len(features), "features, the first is", features[0])
print("tenure" in features, "gender" in features)
for name in features:
    print(" -", name)
first, second, third = features  # unpacking also uses __iter__
print(third)

Notice how little code that is. The value is in the interface: code using `FeatureSet` reads like the question it asks.

> **Check your understanding**
>
> `FeatureSet` defines `__len__` but not `__bool__`. What does this print?
>
> ```python
> no_features = FeatureSet([])
> print("nothing to train on" if not no_features else "ready")
> ```
>
> <details><summary><b>Show answer</b></summary>
>
> `nothing to train on`. Without `__bool__`, Python asks `__len__`, and a length of 0 counts as false. This is the same rule that makes an empty list false.
> </details>

In [ ]:
# Predict the output first, then run this cell
no_features = FeatureSet([])
print("nothing to train on" if not no_features else "ready")

---

## Theory 3 · Equality, arithmetic, and when not to overload

Three more things a confusion matrix should support:

1. **Unpacking.** scikit-learn users write `tn, fp, fn, tp = confusion_matrix(y, p).ravel()`. With `__iter__`, our class supports the same line.
2. **Equality.** Two matrices with the same four counts describe the same result, so `==` should compare counts. Without `__eq__`, Python compares identity (`is`): two separately created objects are never equal.
3. **Addition.** Counts can be pooled. The confusion matrix of two batches together (two cross-validation folds, two months of predictions) is the sum of the two matrices. Here `+` has an obvious meaning, so overloading it is a good idea.

`__add__` returns `NotImplemented` (a special constant, not an error) when it does not know how to add the other object. Python then tries the other object's `__radd__`, and if that also gives up it raises a `TypeError` with a clear message.

This version also gets back the `from_predictions` constructor from notebook 01; we need it in the hands-on parts.

In [ ]:
class ConfusionMatrix:
    """Counts of a binary classifier's hits and misses (1 = positive class)."""

    def __init__(self, tn: int, fp: int, fn: int, tp: int):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    @classmethod
    def from_predictions(cls, y_true, y_pred) -> "ConfusionMatrix":
        t = np.asarray(y_true)  # plain NumPy arrays of 0s and 1s
        p = np.asarray(y_pred)
        # (t == 0) & (p == 0) is True where both hold; .sum() counts the Trues
        return cls(
            tn=int(((t == 0) & (p == 0)).sum()),
            fp=int(((t == 0) & (p == 1)).sum()),
            fn=int(((t == 1) & (p == 0)).sum()),
            tp=int(((t == 1) & (p == 1)).sum()),
        )

    @property
    def n(self) -> int:
        return self.tn + self.fp + self.fn + self.tp

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0

    def __repr__(self) -> str:
        return f"ConfusionMatrix(tn={self.tn}, fp={self.fp}, fn={self.fn}, tp={self.tp})"

    def __iter__(self):
        return iter((self.tn, self.fp, self.fn, self.tp))

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, ConfusionMatrix):
            return NotImplemented
        return tuple(self) == tuple(other)

    def __add__(self, other: object) -> "ConfusionMatrix":
        if not isinstance(other, ConfusionMatrix):
            return NotImplemented
        return ConfusionMatrix(
            tn=self.tn + other.tn,
            fp=self.fp + other.fp,
            fn=self.fn + other.fn,
            tp=self.tp + other.tp,
        )


fold_1 = ConfusionMatrix(tn=450, fp=70, fn=80, tp=105)
fold_2 = ConfusionMatrix(tn=455, fp=62, fn=88, tp=99)
both = fold_1 + fold_2  # __add__
print(both)
print(both == ConfusionMatrix(tn=905, fp=132, fn=168, tp=204))  # __eq__ compares counts
tn, fp, fn, tp = both  # unpacking uses __iter__
print(f"caught {tp} positives, missed {fn}")

In [ ]:
# Adding something that is not a ConfusionMatrix fails loudly, thanks to NotImplemented
try:
    both + 1
except TypeError as exc:
    print("TypeError:", exc)

### A side effect of `__eq__`: the object is no longer hashable

When a class defines `__eq__`, Python sets `__hash__` to `None`. Objects of that class can no longer be dictionary keys or set members:

In [ ]:
try:
    {fold_1}
except TypeError as exc:
    print("TypeError:", exc)

This is deliberate. A dictionary finds keys by their hash, and equal objects must have equal hashes. Our matrix is **mutable**: if you changed `fold_1.tp` after using it as a key, its hash would have to change and the dictionary would lose track of it. The clean fix is to make the class immutable. A frozen dataclass (notebook 07) does exactly that and generates `__eq__` and `__hash__` for you. The finished version in `src/churn/evaluation.py` is such a frozen dataclass.

**Value or identity?** Equality by value is right for *values*: a confusion matrix, a config, a date. It is wrong for *things with identity*, such as a fitted model. Two different models with the same accuracy are not the same model, so a model class should not define `__eq__` on its metrics.

### When not to overload

Special methods make code shorter only when the meaning is obvious. Two rules of thumb:

- **Overload an operator only if every reader would guess the same meaning.** `cm1 + cm2` pooling counts: obvious. `model_a < model_b` comparing models: by accuracy? recall? cost? Readers cannot know. Use an explicit key instead: `best = max(results, key=lambda r: r.recall)`. Here `lambda r: r.recall` is a one-line function that takes a result `r` and returns its recall; `max` calls it on every result and compares those numbers.
- **Keep the usual contracts.** `__len__` returns a non-negative int. `__eq__` is symmetric and returns `NotImplemented` for unknown types. `__repr__` never raises.

---

## Hands-on A · Cross-validation results for the churn model

Now we use these methods on the churn model from the setup. First a worked step: pretend the test customers arrived in two monthly batches, compute one matrix per month, and pool them with `+`. The pooled matrix must be exactly the matrix of all test customers at once.

In [ ]:
half = len(X_test) // 2  # // is division without remainder
# y_test[:half] are the first half of the customers, y_test[half:] the rest
november = ConfusionMatrix.from_predictions(y_test[:half], y_pred[:half])
december = ConfusionMatrix.from_predictions(y_test[half:], y_pred[half:])
print("November:", november)
print("December:", december)
print("Pooled:  ", november + december)
print(
    "Same as all test customers at once?",
    november + december == ConfusionMatrix.from_predictions(y_test, y_pred),
)

### Exercise A.1

*About 15 minutes, including the discussion.*

`cross_val_score` evaluates a model on several folds and returns a bare NumPy array of scores. Wrap it in a `CVResult` class that behaves like a small, readable sequence of fold scores.

**What your code should do.**

- `CVResult(scores, metric="recall")` stores the fold scores (anything list-like, for example the NumPy array from `cross_val_score`) and the name of the metric.
- `len(result)` is the number of folds (here 5); `result[0]` is the score of the first fold (a float).
- `for score in result:` gives the fold scores in order, so `list(result)` equals the list of scores.
- `repr(result)` looks like `CVResult(recall: 0.534 ± 0.021 over 5 folds)`: the mean and the standard deviation of the scores, 3 decimals each.

**pandas and NumPy you need.**

- `cross_val_score(model, X, y, cv=5, scoring="recall")` (already in the starter cell): a NumPy array with one score per fold.
- `list(scores)`: the same numbers as a Python list.
- `np.mean(scores)`, `np.std(scores)`: mean and standard deviation, as floats.
- `f"{value:.3f}"`: a float with 3 decimals.

**Discuss.** `CVResult` only passes calls on to a list, yet it is easier to use than the bare array. Where in your own code would a small wrapper like this help, and when is it just extra code to maintain?

In [ ]:
# Exercise A.1: your code here

from sklearn.model_selection import cross_val_score

fold_scores = cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression()), X_train, y_train, cv=5, scoring="recall"
)


class CVResult:
    def __init__(self, scores, metric: str):
        raise NotImplementedError("Your code here")


result = CVResult(fold_scores, metric="recall")
result

In [ ]:
# Check for exercise A.1: run after solving it
assert len(result) == 5
assert result[0] == fold_scores[0]
assert list(result) == list(fold_scores)
assert repr(result) == (
    f"CVResult(recall: {np.mean(fold_scores):.3f} ± {np.std(fold_scores):.3f} over 5 folds)"
)
print("All checks passed")

<details><summary><b>Show hint</b></summary>

Every method can delegate to the stored scores: `len(self.scores)`, `self.scores[i]`, `iter(self.scores)`. For the repr, use `np.mean` and `np.std` inside an f-string with `:.3f`.
</details>

<details><summary><b>Show solution</b></summary>

```python
class CVResult:
    def __init__(self, scores, metric: str):
        self.scores = list(scores)
        self.metric = metric

    def __len__(self) -> int:
        return len(self.scores)

    def __getitem__(self, fold: int) -> float:
        return self.scores[fold]

    def __iter__(self):
        return iter(self.scores)

    def __repr__(self) -> str:
        return (f"CVResult({self.metric}: {np.mean(self.scores):.3f} "
                f"± {np.std(self.scores):.3f} over {len(self)} folds)")
```

Strictly speaking `__iter__` is optional here: if a class has `__getitem__` with integer indices, `for` falls back to calling it with 0, 1, 2, ... until `IndexError`. Writing `__iter__` explicitly is clearer.
</details>

### Exercise A.2

> If you finish early, try this one.

Cross-validation gives you one confusion matrix per fold, and `+` pools them. Write `pooled_confusion(X, y, n_splits=5)` that trains a fresh model on each training part, computes a `ConfusionMatrix` on each validation part, and returns the pooled matrix. Then compare the **pooled recall** with the **mean of the per-fold recalls**.

**What your code should do.**

- Inputs: `X` a DataFrame of features, `y` a Series of 0/1 labels with the same rows, `n_splits` the number of folds.
- For each fold: fit `make_pipeline(StandardScaler(), LogisticRegression())` on the training rows, predict the validation rows (`.predict`, threshold 0.5), build a `ConfusionMatrix`.
- Return one `ConfusionMatrix`: the sum of all fold matrices. Every customer is validated exactly once, so its `n` equals `len(X)` and `tp + fn` equals the number of churners in `y`.

**pandas and NumPy you need.**

- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y)`: produces one pair `(train_idx, val_idx)` per fold; both are arrays of row **positions**. Loop with `for train_idx, val_idx in skf.split(X, y):`.
- `X.iloc[train_idx]`, `y.iloc[val_idx]`: select rows by position.

**Discuss.** Are the pooled recall and the mean of the fold recalls the same, and why (not)? Which one would you report to the retention team, and which one tells you how stable the model is across folds?

In [ ]:
# Exercise A.2: your code here

from sklearn.model_selection import StratifiedKFold


def pooled_confusion(X: pd.DataFrame, y: pd.Series, n_splits: int = 5) -> ConfusionMatrix:
    raise NotImplementedError("Your code here")


pooled = pooled_confusion(X_train, y_train)
pooled

In [ ]:
# Check for exercise A.2: run after solving it
assert type(pooled) is ConfusionMatrix
assert pooled.n == len(X_train)  # every customer is validated exactly once
assert pooled.tp + pooled.fn == y_train.sum()
print("All checks passed. Pooled recall:", round(pooled.recall, 3))

<details><summary><b>Show hint</b></summary>

Start with `pooled = None`. In the loop, build the fold's matrix `cm` and then write `pooled = cm if pooled is None else pooled + cm`.
</details>

<details><summary><b>Show solution</b></summary>

```python
def pooled_confusion(X: pd.DataFrame, y: pd.Series, n_splits: int = 5) -> ConfusionMatrix:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    pooled = None
    for train_idx, val_idx in skf.split(X, y):
        fold_model = make_pipeline(StandardScaler(), LogisticRegression())
        fold_model.fit(X.iloc[train_idx], y.iloc[train_idx])
        cm = ConfusionMatrix.from_predictions(y.iloc[val_idx], fold_model.predict(X.iloc[val_idx]))
        pooled = cm if pooled is None else pooled + cm
    return pooled
```

The pooled recall and the mean of fold recalls are close but usually not identical. The pooled value weights every churner equally; the mean weights every fold equally. With stratified folds of nearly equal size the difference is tiny. With very unequal batches (say a small December and a large November) it can matter, and pooling is usually the number you want to report.
</details>

---

## Theory 4 · Callable objects and context managers

### `__call__`: objects that behave like functions

Some functions need settings. A cost metric for a classifier, for example, needs to know what a false alarm costs and what a missed case costs, *and* it must accept the usual `(y_true, y_pred)` like any metric function (`f1_score`, `recall_score`). An object with `__call__` keeps the settings as attributes and is then called exactly like a function.

Example: in fraud detection, checking a flagged transaction by hand costs a little, missing a fraud costs a lot.

In [ ]:
class ErrorCost:
    """Total cost of a classifier's mistakes when false alarms and misses have different prices."""

    def __init__(self, false_alarm_cost: float, miss_cost: float):
        self.false_alarm_cost = false_alarm_cost
        self.miss_cost = miss_cost

    def __call__(self, y_true, y_pred) -> float:
        tn, fp, fn, tp = ConfusionMatrix.from_predictions(y_true, y_pred)
        return fp * self.false_alarm_cost + fn * self.miss_cost

    def __repr__(self) -> str:
        return f"ErrorCost(false_alarm_cost={self.false_alarm_cost}, miss_cost={self.miss_cost})"


fraud_cost = ErrorCost(false_alarm_cost=5, miss_cost=200)
truth = [0, 0, 1, 1, 0, 1]
predicted = [0, 1, 1, 0, 0, 1]  # one false alarm, one miss
print(fraud_cost)
print(fraud_cost(truth, predicted))  # called like a function: 1 * 5 + 1 * 200

Because `fraud_cost` is callable, any code written for metric functions accepts it unchanged. A dictionary of metrics can mix plain functions and our object:

In [ ]:
from sklearn.metrics import f1_score, recall_score

metrics = {"recall": recall_score, "f1": f1_score, "cost": fraud_cost}
for name, metric in metrics.items():
    print(f"{name:>6}: {metric(truth, predicted):,.2f}")

You meet this pattern all over the ecosystem: PyTorch layers are called as `layer(x)` (their `__call__` runs `forward`), and scikit-learn scorers made with `make_scorer` are callable objects.

### Context managers: `__enter__` and `__exit__` (brief)

`with` guarantees that set-up and clean-up code runs, even if the block raises an error. You know it from `with open(path) as f:`. Writing one is two methods. A timer is a small, honest use case:

In [ ]:
import time


class Timer:
    """Measure how long the code inside a `with` block takes."""

    def __init__(self, label: str):
        self.label = label
        self.seconds = None

    def __enter__(self) -> "Timer":
        self._start = time.perf_counter()  # a precise clock, in seconds
        return self  # this is what `as t` receives

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.seconds = time.perf_counter() - self._start
        print(f"{self.label}: {self.seconds:.2f} s")
        return False  # do not swallow exceptions raised inside the block


numbers = np.random.default_rng(42).random(1_000_000)  # one million random numbers between 0 and 1
with Timer("sorting one million numbers") as t:
    sorted(numbers)

---

## Hands-on B · A business cost for the retention campaign

Precision and recall do not say what the churn model is worth to the business. Suppose the retention team sends an offer to every customer the model flags:

- each offer costs **50** (discount plus call-centre time),
- a churner who receives the offer stays with probability **0.3**,
- every churner who leaves costs **600** (roughly a year of margin).

These figures are illustrative assumptions; in a real project you get them from the business. Here is the cost as a callable object, built exactly like `ErrorCost` in Theory 4. Read it and run it: it compares "do nothing" with three thresholds.

In [ ]:
class RetentionCost:
    """Total money lost to churn when we act on the model's flags."""

    def __init__(
        self, offer_cost: float = 50.0, customer_value: float = 600.0, acceptance_rate: float = 0.3
    ):
        self.offer_cost = offer_cost
        self.customer_value = customer_value
        self.acceptance_rate = acceptance_rate

    def __call__(self, y_true, y_pred) -> float:
        tn, fp, fn, tp = ConfusionMatrix.from_predictions(y_true, y_pred)
        offers = (tp + fp) * self.offer_cost
        lost_despite_offer = tp * (1 - self.acceptance_rate) * self.customer_value
        lost_unflagged = fn * self.customer_value
        return offers + lost_despite_offer + lost_unflagged

    def __repr__(self) -> str:
        return (
            f"RetentionCost(offer_cost={self.offer_cost}, "
            f"customer_value={self.customer_value}, acceptance_rate={self.acceptance_rate})"
        )


cost = RetentionCost()
print(cost)
# np.zeros_like(y_test): an array of 0s with the same length, i.e. flag nobody
print(f"Do nothing (flag nobody): {cost(y_test, np.zeros_like(y_test)):>10,.0f}")
for threshold in [0.5, 0.3, 0.2]:
    flagged = (churn_proba >= threshold).astype(int)
    print(f"Threshold {threshold}:            {cost(y_test, flagged):>10,.0f}")

### Exercise B.1

*About 10 minutes, including the discussion.*

The three thresholds above were a guess. Write `cheapest_threshold(cost, y_true, proba, thresholds)` that tries every threshold and returns the cheapest one, then use it with `RetentionCost()` on the thresholds 0.05, 0.10, ..., 0.95.

**What your code should do.**

- Inputs: `cost` is any callable metric such as `RetentionCost()`; `y_true` the true 0/1 labels; `proba` a NumPy array of churn probabilities, one per customer (like `churn_proba`); `thresholds` a list of floats.
- For each threshold, flag the customers whose probability is at least the threshold, and compute `cost(y_true, flagged)`.
- Return a tuple `(best_threshold, best_cost)` for the threshold with the lowest cost.

**pandas and NumPy you need.**

- `proba >= 0.3`: an array of `True`/`False`, one per customer; `.astype(int)` turns it into 1/0.
- `[round(0.05 * i, 2) for i in range(1, 20)]` gives the thresholds `0.05, 0.1, ..., 0.95` (already in the starter).

**Discuss.** Here the threshold is picked on the test customers. Why would that make the reported saving too optimistic, and how would you choose it instead? Who in the company should own the three cost assumptions?

In [ ]:
# Exercise B.1: your code here


def cheapest_threshold(cost, y_true, proba, thresholds: list[float]) -> tuple[float, float]:
    raise NotImplementedError("Your code here")


candidates = [round(0.05 * i, 2) for i in range(1, 20)]
best_threshold, best_cost = cheapest_threshold(RetentionCost(), y_test, churn_proba, candidates)
print(f"cheapest threshold: {best_threshold}, cost: {best_cost:,.0f}")

In [ ]:
# Check for exercise B.1: run after solving it
all_costs = [cost(y_test, (churn_proba >= t).astype(int)) for t in candidates]
assert best_threshold in candidates
assert best_cost == min(all_costs)
assert best_cost == cost(y_test, (churn_proba >= best_threshold).astype(int))
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
def cheapest_threshold(cost, y_true, proba, thresholds: list[float]) -> tuple[float, float]:
    best_threshold, best_cost = None, None
    for threshold in thresholds:
        flagged = (proba >= threshold).astype(int)
        current = cost(y_true, flagged)
        if best_cost is None or current < best_cost:
            best_threshold, best_cost = threshold, current
    return best_threshold, best_cost
```

The function never asks *what* `cost` is: anything you can call with `(y_true, y_pred)` works, `RetentionCost()`, `ErrorCost(...)` or a plain function. With these cost assumptions the cheapest threshold is well below 0.5, because missing a churner is much more expensive than an unnecessary offer.
</details>

### Exercise B.2

> **Challenge**

1. Run `sum([november, december])`. It fails, although `november + december` works. Explain why, using the table in Theory 1. (Hint: what is the start value of `sum`?)
2. Write a subclass `SummableConfusionMatrix(ConfusionMatrix)` that adds `__radd__` so that `sum(list_of_matrices)` returns the pooled matrix. Do not change `__add__`.

**What your code should do.**

- `sum([m1, m2, m3])` returns a matrix equal to `m1 + m2 + m3`, and `sum([m1])` returns a matrix equal to `m1`.
- Adding two matrices with `+` keeps working exactly as before.
- No pandas or NumPy needed.

**Discuss.** Our `ConfusionMatrix` defines `__eq__` and is therefore unhashable. How would you make it hashable *correctly*? Why would it be a mistake to give a fitted model class an `__eq__` and `__hash__` based on its accuracy?

In [ ]:
# Exercise B.2: your code here


class SummableConfusionMatrix(ConfusionMatrix):
    def __radd__(self, other):
        raise NotImplementedError("Your code here")


months = [
    SummableConfusionMatrix(tn=500, fp=120, fn=40, tp=110),
    SummableConfusionMatrix(tn=480, fp=130, fn=35, tp=125),
    SummableConfusionMatrix(tn=510, fp=100, fn=50, tp=105),
]
quarter = sum(months)
quarter

In [ ]:
# Check for exercise B.2: run after solving it
assert quarter == ConfusionMatrix(tn=1490, fp=350, fn=125, tp=340)
assert sum(months[:1]) == months[0]
print("All checks passed")

<details><summary><b>Show hint</b></summary>

`sum(items)` computes `0 + items[0] + items[1] + ...`. The first step is `0 + matrix`. `int.__add__` returns `NotImplemented` for a matrix, so Python tries `matrix.__radd__(0)`. Decide what adding 0 should mean.
</details>

<details><summary><b>Show solution</b></summary>

```python
class SummableConfusionMatrix(ConfusionMatrix):
    def __radd__(self, other):
        if other == 0:  # the start value of sum()
            return self
        return self.__add__(other)
```

Alternative without `__radd__`: `sum(months, start=ConfusionMatrix(0, 0, 0, 0))`. That works but puts the burden on every caller.

On hashing: make the matrix immutable (a frozen dataclass, notebook 07) and hash the same fields that `__eq__` compares. Then equal matrices have equal hashes and nobody can change a matrix after it was put in a set. For a fitted model, equality by accuracy would declare two different models (different features, different coefficients) "equal", and a set of models would silently drop one of them. Models have identity, so keep the default identity-based `__eq__` and `__hash__`, and compare models explicitly with `max(models, key=...)`.
</details>

---

## Summary

- Special methods connect your class to Python syntax: `len()`, `[]`, `in`, `for`, `==`, `+`, calling with `()`, `with`.
- Always write `__repr__`. It is the cheapest debugging aid there is.
- Delegate: container methods are often one line that forwards to a list or DataFrame you already hold.
- Return `NotImplemented` (not `False`, not an exception) from `__eq__` and `__add__` for types you do not handle.
- `__call__` turns a configured object into a function: perfect for metrics with business parameters.
- Overload an operator only when its meaning is obvious; otherwise use a named method or a `key=` function.

**Common pitfalls**

- Defining `__eq__` and wondering why the object cannot go into a set (Python removed `__hash__`).
- `__str__` without `__repr__`: lists and error messages still show `<... at 0x...>`.
- Making `+` or `<` mean something surprising.

## Going further

Not part of the live session.

- **`__bool__`** to define truthiness explicitly, **`__contains__`** vs iteration fallback, **`__getitem__` with slices** (`features[0:2]` receives a `slice` object).
- **Comparisons**: `__lt__`, `__le__`, ... and `functools.total_ordering`. Remember "When not to overload" in Theory 3 before using them.
- **`contextlib.contextmanager`** writes a context manager as a generator function with one `yield`, often shorter than a class.
- **`__format__`** controls f-string formatting such as `f"{cm:short}"`.
- The finished `ConfusionMatrix` and `RetentionCost` live in `src/churn/evaluation.py`.

---

## Sources and further reading

**Sources.** This notebook is based on:

- [Python Language Reference, chapter 3: Data model](https://docs.python.org/3/reference/datamodel.html), the complete list of special methods and when Python calls them.
- Python docs: [`contextlib`](https://docs.python.org/3/library/contextlib.html) and [`functools.total_ordering`](https://docs.python.org/3/library/functools.html#functools.total_ordering).
- scikit-learn: [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) (the `ravel()` order we copied) and [Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html) (how scorers use callable objects).

**For self-study.** Where to go next if you want to learn more:

- Luciano Ramalho, *Fluent Python*, 2nd edition, O'Reilly 2022 ([book site](https://www.fluentpython.com/)), part I "Data Structures" and chapter 16 "Operator Overloading" (book, intermediate to advanced): the best explanation of the Python data model there is.
- [Raymond Hettinger, *Python's Class Development Toolkit*](https://www.youtube.com/watch?v=HTLu2DFOdTg) (talk, 45 min), if you have not watched it after notebook 01.

---

[← 02 · Inheritance](02_inheritance.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [04 · Encapsulation and abstraction →](04_encapsulation_and_abstraction.ipynb)